# EE/CS 148B HW2: Colab Setup

This notebook is meant to be imported directly into Colab to provide a quickstart environment setup.


## Colab Setup

Before running:

- Switch the runtime to a GPU runtime.
- We recommend using an A100 runtime for the GRPO experiments.
- Put the `hw2` directory somewhere accessible from Colab, typically Google Drive.

This notebook assumes the repo already exists and only sets up Python dependencies plus the repo import path.

Note: We will be using `pip` for dependencies inside Colab.

In [1]:
%%capture
!pip -q install -U vllm pytest datasets transformers accelerate sentencepiece matplotlib pandas tqdm sympy pylatexenc latex2sympy2_extended "math-verify[antlr4-13-2]"

In [27]:
!pip install einx jaxtyping

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 8.0 MB/s eta 0:00:00


In [2]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
!git clone https://github.com/wduan10/cs148b_hw2.git
%cd cs148b_hw2

Cloning into 'cs148b_hw2'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 56 (delta 10), reused 52 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (56/56), 112.21 KiB | 22.44 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/cs148b_hw2/cs148b_hw2


'/content/cs148b_hw2/cs148b_hw2'

In [18]:
%pwd

'/content/cs148b_hw2'

In [21]:
%ls

alignment/         eecs_148b_hw2.egg-info/  pyproject.toml  tests/
basics/            prepare_submission.sh*   README.md       uv.lock
colab_setup.ipynb  prompt.txt               systems/


In [80]:
!git pull

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.66 KiB | 1.66 MiB/s, done.
From https://github.com/wduan10/cs148b_hw2
   244e9a8..e240937  main       -> origin/main
Updating 244e9a8..e240937
error: Your local changes to the following files would be overwritten by merge:
	systems/attention_benchmark.py
Please commit your changes or stash them before you merge.
Aborting


In [49]:
!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --mode forward-backward \
  --warmup-steps 2 \
  --measure-steps 10

Benchmark results
-----------------
device:        cuda
model size:    large
mode:          forward-backward
bf16:          False
compiled:      False
batch size:    4
context len:   128
warmup steps:  2
measure steps: 10

avg time:      0.153002 s
std time:      0.000706 s


In [50]:
%%bash
wget -q https://developer.nvidia.com/downloads/assets/tools/secure/nsight-systems/2026_1/NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb

apt-get update -yq
apt-get install -yq ./NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb

nsys --version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,599 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:13 https://ppa.launchpadcontent.net/ubuntu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [59]:
!PYTHONPATH=. nsys profile --pytorch=autograd-nvtx --force-overwrite true \
  -o artifacts/nsys_small_ctx128_train \
  python -m systems.benchmark --model-size small --context-length 128 --mode train-step --warmup-steps 5 --measure-steps 10

Benchmark results
-----------------
device:        cuda
model size:    small
mode:          train-step
bf16:          False
compiled:      False
batch size:    4
context len:   128
warmup steps:  5
measure steps: 10

avg time:      0.098613 s
std time:      0.000508 s
Generating '/tmp/nsys-report-8c1a.qdstrm'
[1/1] [========================100%] nsys_small_ctx128_train.nsys-rep
Generated:
	/content/cs148b_hw2/artifacts/nsys_small_ctx128_train.nsys-rep


In [57]:
!PYTHONPATH=. nsys profile --pytorch=autograd-nvtx --force-overwrite true \
  -o artifacts/nsys_small_ctx128_forward_backward \
  python -m systems.benchmark --model-size small --context-length 128 --mode forward-backward --warmup-steps 5 --measure-steps 10

Benchmark results
-----------------
device:        cuda
model size:    small
mode:          forward-backward
bf16:          False
compiled:      False
batch size:    4
context len:   128
warmup steps:  5
measure steps: 10

avg time:      0.105678 s
std time:      0.003480 s
Generating '/tmp/nsys-report-7777.qdstrm'
[1/1] [========================100%] nsys_small_ctx128_forward_backward.nsys-rep
Generated:
	/content/cs148b_hw2/artifacts/nsys_small_ctx128_forward_backward.nsys-rep


In [60]:
from google.colab import files
files.download("/content/cs148b_hw2/artifacts/nsys_small_ctx128_train.nsys-rep")
files.download("/content/cs148b_hw2/artifacts/nsys_small_ctx128_forward_backward.nsys-rep")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [62]:
s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float32)
print(s)

s = torch.tensor(0,dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01,dtype=torch.float16)
print(s)

s = torch.tensor(0,dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01,dtype=torch.float16)
    s += x.type(torch.float32)
print(s)

tensor(10.0001)
tensor(9.9531, dtype=torch.float16)
tensor(10.0021)
tensor(10.0021)


In [71]:
!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --mode forward-backward \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-bf16

Benchmark results
-----------------
device:        cuda
model size:    large
mode:          forward-backward
bf16:          True
compiled:      False
batch size:    4
context len:   128
warmup steps:  5
measure steps: 10

avg time:      0.170059 s
std time:      0.004046 s


In [77]:
!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --context-length 32 \
  --mode forward \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-memory-profiler

!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --context-length 32 \
  --mode forward-backward \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-memory-profiler

!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --context-length 32 \
  --mode train-step \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-memory-profiler

Benchmark results
-----------------
device:        cuda
model size:    large
mode:          forward
bf16:          False
compiled:      False
batch size:    4
context len:   32
warmup steps:  5
measure steps: 10

avg time:      0.087549 s
std time:      0.002774 s
Benchmark results
-----------------
device:        cuda
model size:    large
mode:          forward-backward
bf16:          False
compiled:      False
batch size:    4
context len:   32
warmup steps:  5
measure steps: 10

avg time:      0.192017 s
std time:      0.002976 s
Benchmark results
-----------------
device:        cuda
model size:    large
mode:          train-step
bf16:          False
compiled:      False
batch size:    4
context len:   32
warmup steps:  5
measure steps: 10

avg time:      0.219557 s
std time:      0.001971 s


In [78]:
!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --context-length 128 \
  --mode forward \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-memory-profiler \
  --use-bf16

!PYTHONPATH=. python -m systems.benchmark \
  --model-size large \
  --context-length 128 \
  --mode train-step \
  --warmup-steps 5 \
  --measure-steps 10 \
  --use-memory-profiler \
  --use-bf16

Benchmark results
-----------------
device:        cuda
model size:    large
mode:          forward
bf16:          True
compiled:      False
batch size:    4
context len:   128
warmup steps:  5
measure steps: 10

avg time:      0.099506 s
std time:      0.002361 s
Benchmark results
-----------------
device:        cuda
model size:    large
mode:          train-step
bf16:          True
compiled:      False
batch size:    4
context len:   128
warmup steps:  5
measure steps: 10

avg time:      0.247214 s
std time:      0.004514 s


In [81]:
!PYTHONPATH=. python -m systems.attention_benchmark

{'head_dim': 16, 'sequence_length': 64, 'status': 'ok', 'forward_time_s': 0.016425458998128306, 'backward_time_s': 0.06993797000177437, 'memory_before_backward_mb': 16.50048828125, 'peak_memory_mb': 16.9072265625}
{'head_dim': 16, 'sequence_length': 128, 'status': 'ok', 'forward_time_s': 0.01659819099950255, 'backward_time_s': 0.07200369799829787, 'memory_before_backward_mb': 17.00048828125, 'peak_memory_mb': 18.5634765625}
{'head_dim': 16, 'sequence_length': 256, 'status': 'ok', 'forward_time_s': 0.01982969500022591, 'backward_time_s': 0.06963906900273287, 'memory_before_backward_mb': 18.75048828125, 'peak_memory_mb': 24.8759765625}
{'head_dim': 16, 'sequence_length': 512, 'status': 'ok', 'forward_time_s': 0.02519097000185866, 'backward_time_s': 0.08129962900056853, 'memory_before_backward_mb': 25.25048828125, 'peak_memory_mb': 49.5009765625}
{'head_dim': 16, 'sequence_length': 1024, 'status': 'ok', 'forward_time_s': 0.03902333799851476, 'backward_time_s': 0.12862804100223002, 'memory

`vllm` is installed and used by default for the direct, CoT, and self-consistency GSM8K baselines. If vLLM install or initialization fails in Colab, set `USE_VLLM = False` in the config cell to fall back to HuggingFace generation.


In [ ]:
from pathlib import Path

USE_DRIVE = True
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw2/')  # edit if needed
LOCAL_REPO_ROOT = Path('/content/hw2-sols')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
else:
    REPO_ROOT = LOCAL_REPO_ROOT

assert REPO_ROOT.exists(), f'Repo root does not exist: {REPO_ROOT}'
print('Using repo:', REPO_ROOT)


In [ ]:
import os
import sys

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

In [ ]:
import gc
import json
import random
import subprocess
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from alignment.eval import load_gsm8k_examples, build_prompts, evaluate_vllm
from alignment.prompts import DIRECT_PROMPT_TEMPLATE, COT_PROMPT_TEMPLATE
from alignment.rewards import answer_tag_reward_fn
from alignment.grpo import train_grpo

SEED = 0
set_seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass


# Your code starts here!